# ARC Prize 2026 · ARC-AGI-2 — NVARC test-time training + rule solver

Built on the open-source ARC Prize 2025 winner [NVARC](https://github.com/1ytic/NVARC) (Qwen3-4B fine-tuned on synthetic puzzles, per-task LoRA test-time training, DFS sampling, augmentation scoring). Changes for 2026 (v17: optional NVARC-faithful mode for A/B tests; v16: optional second pass on NVARC's separately trained 2B model; v15: optional second pass, where GPUs left idle after the queue retrain finished tasks with a weak answer on a new example order and colors and decode 16 new views, votes pooled over both adapters; v14: training epochs and learning rate are settings, and 4 copies is a training size; v13: optional extra views in a task's leftover time, off by default; v12: optional 'auto' training size, 16 copies only where they take at most a quarter of the task's budget; commit runs report whether answers are lost in generation or selection; v11: a GPU worker that crashes restarts instead of stopping all four; v10: at most 8 training copies, looser second decode for test outputs with no guess; v7: time-aware test-time training that shrinks or skips training when it would not fit, decoding guaranteed after training, GPU-side scoring patch, verified rule programs take attempt 2, rule solver runs alongside the GPUs):

* **Dynamic per-task time budget.** The hidden set has 240 tasks (2025: 120); NVARC's fixed 1200 s cap would leave the tail of the queue unsolved. Remaining time is split over remaining tasks.
* **Smallest tasks first**, so time saved on easy tasks flows to hard ones.
* **Safety submission first.** A rule-solver `submission.json` is written before the GPU stage, so a crash or timeout still scores.
* **Rule solver takes attempt 2** when it finds a program that reproduces every training pair (97% precise on public training tasks).
* Commit runs (not competition reruns) run a short smoke test on public eval tasks and print a local score.

Inputs: competition `arc-prize-2026-arc-agi-2`, model `sorokin/qwen3_4b_grids15_sft139` (Transformers, bfloat16, v1). Accelerator: GPU L4 x4. Internet: off.

In [ ]:
import time
global_start = time.time()
# Kaggle stops at 12 h. Training no longer runs past the deadline (v7), so 25 min covers
# the final merge and runtime variance (NVARC itself kept 10 min)
global_end_time = global_start + 12 * 3600 - 25 * 60

In [ ]:
TOP_COPIES = '16'  # most test-time training copies per task (time-aware: may use fewer)
TTT_EPOCHS = 1  # test-time training epochs
TTT_LR = '5e-5'  # test-time training learning rate
SECOND_PASS = False  # idle GPUs retrain weak tasks with a second adapter
SECOND_MODEL = False  # second pass on sorokin/qwen3_2b_grids15_sft141 (if attached)
NVARC_MODE = False  # fixed 1200 s per task, input file order, rule solver only fills empty slots
SMOKE_MODE = 'small'  # commit runs only: 'small' (8 smallest, 15 min) or 'realistic'
SMOKE_SEED = 0  # realistic mode: which random sample of public eval tasks
import glob, json, os, subprocess

RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

# prefer the 2026 competition mount even if an older ARC Prize input is still attached
comp_dir = os.path.dirname(find_one("/kaggle/input/*2026*arc-agi-2*/**/arc-agi_test_challenges.json")
                           or find_one("/kaggle/input/**/arc-agi_test_challenges.json"))
# the 4B model by name: with the 2B one also attached, "*grids15*" would sort the 2B first
model_cfg = find_one("/kaggle/input/**/*4b_grids15*/**/config.json") or find_one("/kaggle/input/**/*grids15*/**/config.json")
assert comp_dir and model_cfg, (comp_dir, model_cfg)
model_dir = os.path.dirname(model_cfg)

os.environ["ARC_MODEL_PATH"] = model_dir
second_cfg = find_one("/kaggle/input/**/*2b_grids15*/**/config.json")
if SECOND_PASS and SECOND_MODEL and second_cfg:
    os.environ["ARC_SECOND_MODEL_PATH"] = os.path.dirname(second_cfg)
print("second pass:", SECOND_PASS, "| second model:", os.getenv("ARC_SECOND_MODEL_PATH"))
if RERUN:
    os.environ["ARC_TEST_PATH"] = f"{comp_dir}/arc-agi_test_challenges.json"
else:
    # commit run: smoke test on a few public eval tasks that have solutions
    os.environ["ARC_TEST_PATH"] = f"{comp_dir}/arc-agi_evaluation_challenges.json"
    os.environ["ARC_SOLUTIONS_PATH"] = f"{comp_dir}/arc-agi_evaluation_solutions.json"
    os.environ["ARC_SMOKE_MODE"] = SMOKE_MODE
    if SMOKE_MODE == "realistic":
        # random mixed-size eval tasks on the hidden run's clock per task: 240 tasks share
        # 12 h - 25 min, of which ~8 min is setup, so ~166 s of wall time per task on 4 GPUs
        os.environ["ARC_SMOKE_TASKS"] = os.getenv("ARC_SMOKE_TASKS", "24")
        per_task = (12 * 3600 - 25 * 60 - 8 * 60) / 240
        global_end_time = global_start + 8 * 60 + int(os.environ["ARC_SMOKE_TASKS"]) * per_task
    else:
        os.environ["ARC_SMOKE_TASKS"] = os.getenv("ARC_SMOKE_TASKS", "8")
        # a short clock forces the time-aware training paths (8 copies, base-model decode) to run
        global_end_time = min(global_end_time, global_start + 15 * 60)
os.environ["ARC_TOP_COPIES"] = TOP_COPIES
os.environ["ARC_TTT_EPOCHS"] = str(TTT_EPOCHS)
os.environ["ARC_TTT_LR"] = TTT_LR
os.environ["ARC_SECOND_PASS"] = "1" if SECOND_PASS else "0"
os.environ["ARC_FIXED_BUDGET"] = "1" if NVARC_MODE else "0"
os.environ["ARC_TASK_ORDER"] = "file" if NVARC_MODE else "size"
os.environ["ARC_RULE_SECOND"] = "0" if NVARC_MODE else "1"
os.environ["ARC_LLM_DIR"] = "../inference_outputs"
os.environ["ARC_SUBMISSION_PATH"] = "/kaggle/working/submission.json"
print("rerun:", RERUN, "| model:", model_dir, "| test:", os.environ["ARC_TEST_PATH"])

In [ ]:
!pip uninstall -y tensorflow > /dev/null 2>&1

In [ ]:
# NVARC's flash-attention patch for batched decoding in unsloth's Qwen3 fast path (idempotent)
patch = b'@@ -272,7 +277,7 @@\r\n \r\n         # Mistral Nemo 12b has weird dimensions\r\n         if attention_size != hidden_size:\r\n-            self.temp_O = torch.empty((1, bsz, hidden_size), dtype = dtype, device = device)\r\n+            self.temp_O = torch.empty((bsz, 1, hidden_size), dtype = dtype, device = device)\r\n         else:\r\n             self.temp_O = self.temp_QA[1][:,:,:hidden_size]\r\n         pass\r\n@@ -333,52 +338,12 @@\r\n     Kn = self.paged_attention_K[:kv_seq_len].permute(1, 2, 0, 3)\r\n     Vn = self.paged_attention_V[:kv_seq_len].permute(1, 2, 0, 3)\r\n \r\n-    # Handle sliding windows\r\n-    sliding_window = getattr(self.config, "sliding_window", None)\r\n-    if sliding_window is not None and kv_seq_len > sliding_window:\r\n-        # From https://github.com/huggingface/transformers/blob/main/src/transformers/models/mistral/modeling_mistral.py#L193\r\n-        slicing_tokens = 1 - sliding_window\r\n-        Knn = Kn[:, :, slicing_tokens:, :]#.contiguous()\r\n-        Vnn = Vn[:, :, slicing_tokens:, :]#.contiguous()\r\n-    else:\r\n-        Knn, Vnn = Kn, Vn\r\n-    pass\r\n+    Qnn = Qn.transpose(1, 2)\r\n+    Knn = Kn.transpose(1, 2)\r\n+    Vnn = Vn.transpose(1, 2)\r\n \r\n-    # when qlen==vlen and attn_mask is None, we should use causal attention\r\n-    Q_len = Qn.shape[-2]\r\n-    K_len = Knn.shape[-2]\r\n-    if attention_mask is None and Q_len == K_len:\r\n-        is_causal = True\r\n-    else:\r\n-        is_causal = False\r\n+    A = flash_attn_func(Qnn, Knn, Vnn)\r\n \r\n-    # Grouped query attention\r\n-    _, _, cached_len, _ = Knn.shape\r\n-    if bsz == 1 or not SDPA_HAS_GQA and n_groups != 1:\r\n-        Knn = Knn[:, :, None, :, :].expand(bsz, n_kv_heads, n_groups, cached_len, head_dim)\r\n-        Vnn = Vnn[:, :, None, :, :].expand(bsz, n_kv_heads, n_groups, cached_len, head_dim)\r\n-        Knn = Knn.reshape(bsz, n_heads, cached_len, head_dim)\r\n-        Vnn = Vnn.reshape(bsz, n_heads, cached_len, head_dim)\r\n-    pass\r\n-    # else:\r\n-    #     Knn, Vnn = Knn, Vnn\r\n-    # pass\r\n-\r\n-    # Attention\r\n-    if bsz == 1:\r\n-        Qn *= self.scalar # See https://github.com/ggerganov/llama.cpp/issues/7805#issuecomment-2153349963\r\n-        # It seems like doing (Q * scalar) @ K is better than (Q @ K) * scalar to stop overflows\r\n-        A = torch_matmul(Qn, Knn.transpose(2, 3), out = self.attention[:,:,:,:cached_len])\r\n-        # if attention_mask is not None: A += attention_mask # Must add attention_mask for batched\r\n-        A[:] = torch_nn_functional_softmax(A, dim = -1, dtype = torch.float32)#.to(A.dtype)\r\n-        A = torch_matmul(A, Vnn, out = Qn)\r\n-    else:\r\n-        if SDPA_HAS_GQA:\r\n-            A = scaled_dot_product_attention(Qn, Knn, Vnn, attn_mask = attention_mask, is_causal = is_causal, enable_gqa = True)\r\n-        else:\r\n-            A = scaled_dot_product_attention(Qn, Knn, Vnn, attn_mask = attention_mask, is_causal = is_causal)\r\n-    pass\r\n-    A = A.transpose(1, 2)\r\n     A = A.reshape(bsz, 1, attention_size)\r\n     A = fast_linear_forward(self.o_proj, A, out = self.temp_O)\r\n     return A, (Kn, Vn)\r\n'
import importlib.util
# locate unsloth without importing it (importing would grab a CUDA context in this kernel)
target = os.path.join(os.path.dirname(importlib.util.find_spec("unsloth").origin), "models", "qwen3.py")
if "flash_attn_func(Qnn, Knn, Vnn)" in open(target).read():
    print("unsloth qwen3 patch already applied")
else:
    open("qwen3.patch", "wb").write(patch)
    print(subprocess.run(["patch", "--binary", target, "qwen3.patch"], capture_output=True, text=True).stdout)

In [ ]:
%%writefile arc_solver.py
"""Program-search solver for ARC-AGI tasks.

Each generator looks at the training pairs and proposes candidate programs
(grid -> grid functions). A program is kept only if it reproduces every
training output exactly; the first two distinct test predictions become
attempt_1 and attempt_2. Generators are ordered from simplest to most
flexible, so simpler explanations win ties.
"""
import time
from collections import Counter

import numpy as np

# ---------------------------------------------------------------- utilities

DIHEDRAL = [
    ("id", lambda g: g),
    ("rot90", lambda g: np.rot90(g, 1)),
    ("rot180", lambda g: np.rot90(g, 2)),
    ("rot270", lambda g: np.rot90(g, 3)),
    ("flipud", lambda g: g[::-1]),
    ("fliplr", lambda g: g[:, ::-1]),
    ("transpose", lambda g: g.T),
    ("antitranspose", lambda g: np.rot90(g, 2).T),
]


def background(g):
    vals, counts = np.unique(g, return_counts=True)
    return int(vals[np.argmax(counts)])


def components(g, bg, diag=True, multicolor=False):
    """Connected components of non-background cells: list of (mask, color_set)."""
    h, w = g.shape
    seen = np.zeros_like(g, dtype=bool)
    out = []
    nbrs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if diag:
        nbrs += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    for r in range(h):
        for c in range(w):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = g[r, c]
            stack = [(r, c)]
            seen[r, c] = True
            cells = []
            while stack:
                y, x = stack.pop()
                cells.append((y, x))
                for dy, dx in nbrs:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and g[ny, nx] != bg \
                            and (multicolor or g[ny, nx] == col):
                        seen[ny, nx] = True
                        stack.append((ny, nx))
            m = np.zeros_like(g, dtype=bool)
            ys, xs = zip(*cells)
            m[list(ys), list(xs)] = True
            out.append(m)
    return out


def bbox(mask):
    ys, xs = np.nonzero(mask)
    return ys.min(), ys.max() + 1, xs.min(), xs.max() + 1


def learn_colormap(pairs):
    """Consistent per-pixel color map a->b over same-shape pairs, or None."""
    m = {}
    for a, b in pairs:
        if a.shape != b.shape:
            return None
        for x, y in zip(a.ravel().tolist(), b.ravel().tolist()):
            if m.setdefault(x, y) != y:
                return None
    return m


def apply_colormap(g, m):
    out = g.copy()
    for a, b in m.items():
        out[g == a] = b
    return out


# --------------------------------------------------------------- generators
# Each generator: gen(train) -> iterable of (name, fn). train is a list of
# (input ndarray, output ndarray). Programs are verified by the caller.

def gen_dihedral(train):
    for name, f in DIHEDRAL:
        yield name, f


def gen_dihedral_colormap(train):
    for name, f in DIHEDRAL:
        try:
            m = learn_colormap([(f(a), b) for a, b in train])
        except Exception:
            m = None
        if m:
            yield f"{name}+cmap", (lambda g, f=f, m=m: apply_colormap(f(g), m))


def gen_constant(train):
    outs = [b for _, b in train]
    if all(o.shape == outs[0].shape and (o == outs[0]).all() for o in outs):
        yield "constant", (lambda g, o=outs[0]: o.copy())


def gen_upscale(train):
    a, b = train[0]
    if b.shape[0] % a.shape[0] or b.shape[1] % a.shape[1]:
        return
    ky, kx = b.shape[0] // a.shape[0], b.shape[1] // a.shape[1]
    for name, f in DIHEDRAL:
        yield f"upscale{ky}x{kx}+{name}", (lambda g, f=f: np.kron(f(g), np.ones((ky, kx), dtype=g.dtype)))
    # fractal: each input cell becomes a copy of the input (or blank)
    if (ky, kx) == a.shape:
        bg = background(a)
        yield "fractal", (lambda g: np.kron(g != bg, np.ones_like(g)) * np.tile(g, g.shape) + bg * (1 - np.kron(g != bg, np.ones_like(g))))
        yield "fractal_inv", (lambda g: np.kron(g == bg, np.ones_like(g)) * np.tile(g, g.shape) + bg * (1 - np.kron(g == bg, np.ones_like(g))))


def gen_tile(train):
    """Output is a ky x kx tiling of transformed copies of the input."""
    a, b = train[0]
    if b.shape[0] % a.shape[0] or b.shape[1] % a.shape[1]:
        return
    ky, kx = b.shape[0] // a.shape[0], b.shape[1] // a.shape[1]
    if ky * kx == 1:
        return
    # choose, per tile slot, the dihedral transform consistent with all pairs
    choice = {}
    for ty in range(ky):
        for tx in range(kx):
            for name, f in DIHEDRAL:
                ok = True
                for x, y in train:
                    h, w = x.shape
                    if y.shape != (h * ky, w * kx):
                        return
                    fx = f(x)
                    if fx.shape != (h, w) or not (y[ty * h:(ty + 1) * h, tx * w:(tx + 1) * w] == fx).all():
                        ok = False
                        break
                if ok:
                    choice[(ty, tx)] = f
                    break
            else:
                return

    def run(g):
        h, w = g.shape
        out = np.zeros((h * ky, w * kx), dtype=g.dtype)
        for (ty, tx), f in choice.items():
            out[ty * h:(ty + 1) * h, tx * w:(tx + 1) * w] = f(g)
        return out
    yield f"tile{ky}x{kx}", run


def gen_downscale(train):
    a, b = train[0]
    if a.shape[0] % b.shape[0] or a.shape[1] % b.shape[1]:
        return
    ky, kx = a.shape[0] // b.shape[0], a.shape[1] // b.shape[1]
    if ky * kx == 1:
        return

    def blocks(g):
        h, w = g.shape[0] // ky, g.shape[1] // kx
        return g[:h * ky, :w * kx].reshape(h, ky, w, kx).transpose(0, 2, 1, 3).reshape(h, w, ky * kx)

    yield "subsample", (lambda g: g[::ky, ::kx])

    def majority(g):
        bl = blocks(g)
        return np.apply_along_axis(lambda v: np.bincount(v, minlength=10).argmax(), 2, bl)

    def anycolor(g):
        bg = background(g)
        bl = blocks(g)
        return np.apply_along_axis(lambda v: next((c for c in v if c != bg), bg), 2, bl)
    yield "block_majority", majority
    yield "block_any", anycolor


def _object_selectors():
    """Rules that pick one object from a list of (mask, grid) by a property."""
    def size(m, g): return m.sum()
    def area(m, g):
        y0, y1, x0, x1 = bbox(m)
        return (y1 - y0) * (x1 - x0)
    def ncolors(m, g): return len(np.unique(g[m]))
    # position-based (top/left) and density selectors fit training pairs by
    # coincidence too often: 5 of the 6 wrong programs on the training set
    props = [("size", size), ("area", area), ("ncolors", ncolors)]
    sels = []
    for pname, p in props:
        sels.append((f"max_{pname}", p, max))
        sels.append((f"min_{pname}", p, min))
    return sels


def _color_rarity_select(objs, g, rarest):
    cols = [Counter(g[m].tolist()).most_common(1)[0][0] for m in objs]
    cnt = Counter(cols)
    target = [c for c in cnt if cnt[c] == (min if rarest else max)(cnt.values())]
    cand = [m for m, c in zip(objs, cols) if c in target]
    return cand[0] if len(cand) == 1 else None


def _shape_key(m, g):
    y0, y1, x0, x1 = bbox(m)
    return (m[y0:y1, x0:x1].tobytes(), m[y0:y1, x0:x1].shape)


def gen_crop(train):
    """Output is the bounding-box crop of one selected object (or of all content)."""
    def crop_all(g):
        bg = background(g)
        y0, y1, x0, x1 = bbox(g != bg)
        return g[y0:y1, x0:x1]
    yield "crop_content", crop_all

    for diag in (True, False):
        for multi in (True, False):
            for sname, p, agg in _object_selectors():
                def run(g, diag=diag, multi=multi, p=p, agg=agg, masked=False):
                    bg = background(g)
                    objs = components(g, bg, diag, multi)
                    if not objs:
                        return None
                    vals = [p(m, g) for m in objs]
                    best = agg(vals)
                    pick = [m for m, v in zip(objs, vals) if v == best]
                    if len(pick) != 1:
                        return None
                    y0, y1, x0, x1 = bbox(pick[0])
                    return g[y0:y1, x0:x1]
                yield f"crop_{sname}_d{int(diag)}m{int(multi)}", run
            for rarest in (True, False):
                def run2(g, diag=diag, multi=multi, rarest=rarest):
                    bg = background(g)
                    objs = components(g, bg, diag, multi)
                    m = _color_rarity_select(objs, g, rarest) if objs else None
                    if m is None:
                        return None
                    y0, y1, x0, x1 = bbox(m)
                    return g[y0:y1, x0:x1]
                yield f"crop_{'rare' if rarest else 'common'}color_d{int(diag)}m{int(multi)}", run2

            def run3(g, diag=diag, multi=multi):
                # the object whose shape is unique among the objects
                bg = background(g)
                objs = components(g, bg, diag, multi)
                keys = [_shape_key(m, g) for m in objs]
                cnt = Counter(keys)
                pick = [m for m, k in zip(objs, keys) if cnt[k] == 1]
                if len(pick) != 1:
                    return None
                y0, y1, x0, x1 = bbox(pick[0])
                return g[y0:y1, x0:x1]
            yield f"crop_uniqueshape_d{int(diag)}m{int(multi)}", run3


def _split_parts(g):
    """Split a grid into cells separated by full-length single-color lines."""
    h, w = g.shape
    rows = [r for r in range(h) if len(set(g[r].tolist())) == 1]
    cols = [c for c in range(w) if len(set(g[:, c].tolist())) == 1]
    sep_colors = set(g[r, 0] for r in rows) | set(g[0, c] for c in cols)
    # keep only separators of a color that never appears as a 'content' line everywhere
    parts = []
    if rows or cols:
        for sc in sep_colors:
            rr = [-1] + [r for r in rows if g[r, 0] == sc] + [h]
            cc = [-1] + [c for c in cols if g[0, c] == sc] + [w]
            ps = []
            for i in range(len(rr) - 1):
                for j in range(len(cc) - 1):
                    if rr[i + 1] - rr[i] > 1 and cc[j + 1] - cc[j] > 1:
                        ps.append(g[rr[i] + 1:rr[i + 1], cc[j] + 1:cc[j + 1]])
            if len(ps) >= 2 and len(set(p.shape for p in ps)) == 1:
                parts.append(ps)
    return parts


def _halves(g, n, axis):
    if g.shape[axis] % n:
        return None
    return np.split(g, n, axis=axis)


def gen_split_logic(train):
    """Output combines equally sized parts of the input with a boolean op."""
    out_colors = set()
    for _, b in train:
        out_colors |= set(np.unique(b).tolist())

    def part_lists(g):
        res = list(_split_parts(g))
        for n in (2, 3):
            for ax in (0, 1):
                p = _halves(g, n, ax)
                if p:
                    res.append(p)
        return res

    ops = {
        "and": lambda m: np.logical_and.reduce(m),
        "or": lambda m: np.logical_or.reduce(m),
        "xor": lambda m: np.logical_xor.reduce(m),
        "nor": lambda m: ~np.logical_or.reduce(m),
        "nand": lambda m: ~np.logical_and.reduce(m),
        "a_not_b": lambda m: m[0] & ~np.logical_or.reduce(m[1:]),
        "b_not_a": lambda m: m[-1] & ~np.logical_or.reduce(m[:-1]),
    }
    n_ways = len(part_lists(train[0][0]))
    for way in range(n_ways):
        for opname, op in ops.items():
            for fg in out_colors:
                for bgc in out_colors:
                    if fg == bgc:
                        continue

                    def run(g, way=way, op=op, fg=fg, bgc=bgc):
                        pl = part_lists(g)
                        if way >= len(pl):
                            return None
                        parts = pl[way]
                        bg = Counter(np.concatenate([p.ravel() for p in parts]).tolist()).most_common(1)[0][0]
                        m = op([p != bg for p in parts])
                        return np.where(m, fg, bgc)
                    yield f"split{way}_{opname}_{fg}_{bgc}", run

        # overlay parts in priority order (later parts drawn on top of earlier ones)
        for order in ("fwd", "rev"):
            def run_ov(g, way=way, order=order):
                pl = part_lists(g)
                if way >= len(pl):
                    return None
                parts = pl[way] if order == "fwd" else pl[way][::-1]
                bg = Counter(np.concatenate([p.ravel() for p in parts]).tolist()).most_common(1)[0][0]
                out = parts[0].copy()
                for p in parts[1:]:
                    out[p != bg] = p[p != bg]
                return out
            yield f"split{way}_overlay_{order}", run_ov

        def run_odd(g, way=way):
            # the one part that differs from the others
            pl = part_lists(g)
            if way >= len(pl):
                return None
            keys = [p.tobytes() for p in pl[way]]
            cnt = Counter(keys)
            pick = [p for p, k in zip(pl[way], keys) if cnt[k] == 1]
            return pick[0] if len(pick) == 1 else None
        yield f"split{way}_oddone", run_odd


def gen_gravity(train):
    def make(direction):
        def run(g):
            bg = background(g)
            k = {"down": 0, "left": 1, "up": 2, "right": 3}[direction]
            r = np.rot90(g, k).copy()
            for c in range(r.shape[1]):
                col = r[:, c]
                vals = col[col != bg]
                r[:, c] = np.concatenate([np.full(len(col) - len(vals), bg), vals])
            return np.rot90(r, -k)
        return run
    for d in ("down", "up", "left", "right"):
        yield f"gravity_{d}", make(d)


def _enclosed(g, bg):
    """Background cells not 4-connected to the border."""
    h, w = g.shape
    reach = np.zeros_like(g, dtype=bool)
    stack = [(r, c) for r in range(h) for c in (0, w - 1)] + [(r, c) for c in range(w) for r in (0, h - 1)]
    while stack:
        y, x = stack.pop()
        if 0 <= y < h and 0 <= x < w and not reach[y, x] and g[y, x] == bg:
            reach[y, x] = True
            stack += [(y + 1, x), (y - 1, x), (y, x + 1), (y, x - 1)]
    return (g == bg) & ~reach


def gen_fill_enclosed(train):
    cols = set()
    for a, b in train:
        if a.shape != b.shape:
            return
        cols |= set(b[a != b].tolist())
    for c in cols:
        def run(g, c=c):
            out = g.copy()
            out[_enclosed(g, background(g))] = c
            return out
        yield f"fill_enclosed_{c}", run


# local cellular-automaton style rules: out[y,x] = table[features(y,x)]

def _neigh_features(kind):
    def feat(g, bg):
        h, w = g.shape
        p = np.pad(g, 2, constant_values=-1)
        F = []
        for y in range(h):
            for x in range(w):
                Y, X = y + 2, x + 2
                if kind == "c":
                    f = (p[Y, X],)
                elif kind == "c4":
                    f = (p[Y, X], p[Y - 1, X], p[Y + 1, X], p[Y, X - 1], p[Y, X + 1])
                elif kind == "c8":
                    f = tuple(p[Y - 1:Y + 2, X - 1:X + 2].ravel())
                elif kind == "c24":
                    f = tuple(p[Y - 2:Y + 3, X - 2:X + 3].ravel())
                elif kind == "rowcol":
                    # center + set of colors in row and column
                    f = (p[Y, X], frozenset(g[y].tolist()) - {bg}, frozenset(g[:, x].tolist()) - {bg})
                elif kind == "rays":
                    # center + first non-bg color seen in each of the 4 directions
                    def ray(dy, dx):
                        yy, xx = y + dy, x + dx
                        while 0 <= yy < h and 0 <= xx < w:
                            if g[yy, xx] != bg:
                                return g[yy, xx]
                            yy += dy
                            xx += dx
                        return -1
                    f = (p[Y, X], ray(-1, 0), ray(1, 0), ray(0, -1), ray(0, 1))
                F.append(f)
        return F
    return feat


def gen_local_rules(train):
    if any(a.shape != b.shape for a, b in train):
        return
    for kind in ("c", "c4", "c8", "rays", "rowcol", "c24"):
        feat = _neigh_features(kind)
        table = {}
        ok = True
        for a, b in train:
            bg = background(a)
            for f, v in zip(feat(a, bg), b.ravel().tolist()):
                if table.setdefault(f, v) != v:
                    ok = False
                    break
            if not ok:
                break
        if not ok:
            continue

        def run(g, feat=feat, table=table):
            bg = background(g)
            vals = []
            for f in feat(g, bg):
                if f not in table:
                    return None
                vals.append(table[f])
            return np.array(vals).reshape(g.shape)
        yield f"local_{kind}", run


def gen_object_recolor(train):
    """Each object keeps its shape but is recolored by a property (size, shape, border contact, ...)."""
    if any(a.shape != b.shape for a, b in train):
        return

    def touches(m):
        return bool(m[0].any() or m[-1].any() or m[:, 0].any() or m[:, -1].any())

    def rank(objs, f):
        vals = sorted(set(f(m) for m in objs))
        return lambda m: vals.index(f(m))

    props = {
        "size": lambda objs: (lambda m: int(m.sum())),
        "size_rank": lambda objs: rank(objs, lambda m: int(m.sum())),
        "size_rank_desc": lambda objs: (lambda r: (lambda m: -r(m)))(rank(objs, lambda m: -int(m.sum()))),
        "shape": lambda objs: (lambda m: _shape_key(m, None)),
        "border": lambda objs: touches,
        "height": lambda objs: (lambda m: bbox(m)[1] - bbox(m)[0]),
        "width": lambda objs: (lambda m: bbox(m)[3] - bbox(m)[2]),
        "holes": lambda objs: (lambda m: int(_enclosed(np.where(m, 1, 0), 0).sum() > 0)),
    }
    for diag in (False, True):
        for pname, pf in props.items():
            table = {}
            ok = True
            for a, b in train:
                bg = background(a)
                if not ((a == b) | (a != bg)).all():   # only object cells may change
                    ok = False
                    break
                objs = components(a, bg, diag, False)
                f = pf(objs)
                for m in objs:
                    cols = np.unique(b[m])
                    if len(cols) != 1 or table.setdefault(f(m), int(cols[0])) != int(cols[0]):
                        ok = False
                        break
                if not ok:
                    break
            if not ok or not table:
                continue

            def run(g, diag=diag, pf=pf, table=table):
                bg = background(g)
                objs = components(g, bg, diag, False)
                f = pf(objs)
                out = g.copy()
                for m in objs:
                    if f(m) not in table:
                        return None
                    out[m] = table[f(m)]
                return out
            yield f"recolor_{pname}_d{int(diag)}", run


def gen_compose(train):
    """A shape-changing base program followed by a dihedral transform and/or a color map."""
    for base_gen in (gen_crop, gen_split_logic, gen_downscale, gen_upscale):
        for bname, bfn in base_gen(train):
            mids = []
            for a, _ in train:
                m = _safe(bfn, a)
                if m is None:
                    break
                mids.append(m)
            if len(mids) != len(train):
                continue
            for pname, pf in DIHEDRAL[1:]:
                if all(pf(m).shape == b.shape for m, (_, b) in zip(mids, train)):
                    cm = learn_colormap([(pf(m), b) for m, (_, b) in zip(mids, train)])
                    if cm is not None:
                        yield f"{bname}>{pname}", (lambda g, bfn=bfn, pf=pf: pf(bfn(g)))
                        if any(k != v for k, v in cm.items()):
                            yield f"{bname}>{pname}+cmap", (lambda g, bfn=bfn, pf=pf, cm=cm: apply_colormap(pf(bfn(g)), cm))
            if all(m.shape == b.shape for m, (_, b) in zip(mids, train)):
                cm = learn_colormap([(m, b) for m, (_, b) in zip(mids, train)])
                if cm and any(k != v for k, v in cm.items()):
                    yield f"{bname}+cmap", (lambda g, bfn=bfn, cm=cm: apply_colormap(bfn(g), cm))


GENERATORS = [
    gen_constant,
    gen_dihedral,
    gen_dihedral_colormap,
    gen_upscale,
    gen_tile,
    gen_downscale,
    gen_crop,
    gen_split_logic,
    gen_gravity,
    gen_fill_enclosed,
    gen_object_recolor,
    gen_local_rules,
    gen_compose,
]


# ------------------------------------------------------------------- driver

def _safe(fn, g):
    try:
        r = fn(g)
    except Exception:
        return None
    if r is None:
        return None
    r = np.asarray(r)
    if r.ndim != 2 or r.size == 0 or r.shape[0] > 30 or r.shape[1] > 30:
        return None
    return r.astype(int)


def _plausible(pred, test_in, train):
    """Reject degenerate test predictions that no training output resembles."""
    if pred.shape == test_in.shape and (pred == test_in).all() and \
            not any(a.shape == b.shape and (a == b).all() for a, b in train):
        return False
    if len(np.unique(pred)) == 1 and all(len(np.unique(b)) > 1 for _, b in train):
        return False
    return True


def verified_candidates(task, time_budget=20.0):
    """Programs that reproduce every training pair: (names, per-test list of predictions)."""
    t0 = time.time()
    train = [(np.array(p["input"]), np.array(p["output"])) for p in task["train"]]
    tests = [np.array(t["input"]) for t in task["test"]]
    found = []  # (name, [prediction per test])
    for gen in GENERATORS:
        if time.time() - t0 > time_budget or len(found) >= 2:
            break
        try:
            programs = list(gen(train))
        except Exception:
            continue
        for name, fn in programs:
            if time.time() - t0 > time_budget:
                break
            if all((r := _safe(fn, a)) is not None and r.shape == b.shape and (r == b).all() for a, b in train):
                preds = [_safe(fn, t) for t in tests]
                if all(p is not None and _plausible(p, t, train) for p, t in zip(preds, tests)):
                    key = tuple(p.tobytes() + bytes(p.shape) for p in preds)
                    if all(key != k for _, _, k in found):
                        found.append((name, preds, key))
                        if len(found) >= 2:
                            break
    names = [f[0] for f in found]
    return names, [[f[1][i].tolist() for f in found] for i in range(len(tests))]


def solve_task(task, time_budget=20.0, return_names=False):
    names, per_test = verified_candidates(task, time_budget)
    attempts = []
    for cands, t in zip(per_test, task["test"]):
        fallback = t["input"]
        a1 = cands[0] if cands else fallback
        a2 = cands[1] if len(cands) > 1 else (fallback if a1 != fallback else [[0]])
        attempts.append([a1, a2])
    return (attempts, names) if return_names else attempts


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


# v15 diagnostic only: kgmon with votes weighted half and double against the augmentation score
def score_kgmon_half(guesses):
    return score_sum(guesses, lambda g: 0.5 * len(g) - np.mean([np.mean(x["score_aug"]) for x in g]))

def score_kgmon_double(guesses):
    return score_sum(guesses, lambda g: 2 * len(g) - np.mean([np.mean(x["score_aug"]) for x in g]))


# v17 diagnostic only: votes alone, and augmentation score alone
def score_votes(guesses):
    return score_sum(guesses, lambda g: len(g) - 1e-3 * np.mean([np.mean(x["score_aug"]) for x in g]))

def score_aug_only(guesses):
    return score_sum(guesses, lambda g: -np.mean([np.mean(x["score_aug"]) for x in g]))


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
    score_kgmon_half,
    score_kgmon_double,
    score_votes,
    score_aug_only,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile llm_worker.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # perf patch (mikelou1, Apache 2.0): same NLL, but only the 12 ARC-token columns leave the GPU
    logits_f = logits.float()
    token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=logits.device)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - logits_f.index_select(-1, token_ids)
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    # perf patch: teacher-forced scoring needs no KV cache, and only target-token log-probs leave the GPU
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        positions = torch.arange(query_length - 1, query_length - 1 + len(answer_tokens), device=model.device)
        targets = torch.tensor(answer_tokens, dtype=torch.long, device=model.device)
        answer_score = (batch_logits[row_id, positions, targets] - batch_log_norm[row_id, positions]).sum()
        result.append(-answer_score.item())
    return result


def task_budget(queue, n_workers, end_time):
    """Seconds this worker may spend on its next task.

    The 2026 hidden test set has 240 tasks (2025 had 120), so NVARC's fixed
    1200s cap would run out of time with tasks still queued. Instead, split
    the remaining time evenly over the tasks this worker is still expected to
    take (the queue also holds one None sentinel per worker).
    """
    if FIXED_BUDGET:  # v17 NVARC-faithful mode: NVARC's fixed per-task cap
        return MAX_TASK_SECONDS
    remaining_tasks = max(0, queue.qsize() - n_workers)
    my_share = remaining_tasks / n_workers + 1
    budget = (end_time - time.time()) / my_share
    return max(MIN_TASK_SECONDS, min(MAX_TASK_SECONDS, budget))


def plan_training(budget, time_left, tokens_16, sec_per_token, top=16, share_16=None):
    """How many augmented copies to train on for this task: top (4, 8, 16 or 24), 16, 8, 4, or 0
    (decode with the base model).

    Training cannot be interrupted, so near the end of the run a large task could train past the
    deadline and come back with nothing. With a measured training speed (seconds per training
    token, from this worker's earlier tasks), pick the largest set that still leaves time to
    decode. tokens_16 is the token count of the 16-copy set; other sizes scale linearly.
    share_16 (v12 "auto"): 16 copies only when their training takes at most this share of the
    task's budget, since decoding, not training, uses most of a task's time.
    """
    n = top if budget >= 600 else min(top, 8)
    if sec_per_token is None:
        return min(n, 16)  # first task: no estimate yet, keep the old rule
    for m in [c for c in (24, 16, 8, 4) if c <= n]:
        est = sec_per_token * tokens_16 * m / 16
        if m > 16:
            # copies beyond NVARC's 16 only when they fit comfortably in this task's budget
            fits_task = est <= 0.6 * budget
        elif m == 16 and share_16:
            fits_task = est <= share_16 * budget
        else:
            fits_task = budget > SCARCE_BUDGET or est <= 0.5 * budget
        if est + MIN_DECODE_SECONDS <= time_left and fits_task:
            return m
    return 0


MIN_TASK_SECONDS = 120
MAX_TASK_SECONDS = 1200
MIN_DECODE_SECONDS = 240
# below this per-task budget time is short, and training must fit in half of it
SCARCE_BUDGET = 300
# looser second pass for a test output that got no guess: 2% total probability instead of 20%
FALLBACK_MAX_SCORE = -np.log(0.02)
FALLBACK_DECODE_SECONDS = 120
# ARC_TOP_COPIES=auto: 16-copy training may take at most this share of a task's budget (else 8)
AUTO_SHARE_16 = 0.25
# v13: decode a second set of 16 views while the task's budget has at least this much time left.
# Off by default: loop 5 scored 11.0/24 with them vs 11.5 without, and used the whole clock
EXTRA_VIEWS = os.environ.get("ARC_EXTRA_VIEWS", "0") == "1"
EXTRA_VIEWS_MIN_SECONDS = 60
# v14: test-time training intensity (NVARC: 1 epoch at 5e-5)
TTT_EPOCHS = int(os.environ.get("ARC_TTT_EPOCHS", "1"))
TTT_LR = float(os.environ.get("ARC_TTT_LR", "5e-5"))
# v15: once the queue is empty, an idle GPU trains a second adapter (new example order and colors)
# on a finished task whose best answer is weak, and decodes 16 new views with it; the merge votes
# over both passes. Every realistic test so far ended with the GPUs idle for ~24% of the clock, and
# more views from the same adapter found nothing new (loop 5), so the second pass varies the adapter
SECOND_PASS = os.environ.get("ARC_SECOND_PASS", "0") == "1"
# a test output whose top answer has fewer votes (decoded beams) than this counts as weak
SECOND_PASS_MAX_VOTES = 8
# v17: NVARC-faithful mode uses NVARC's fixed 1200 s per task instead of the dynamic share. Plain NVARC
# reproductions score 32.22 on the public leaderboard, above our v1 (30.14) and v7 (30.56)
FIXED_BUDGET = os.environ.get("ARC_FIXED_BUDGET", "0") == "1"


def worker(rank, queue, end_time, n_workers=4):

    model_path = os.environ["ARC_MODEL_PATH"]
    test_path = os.environ["ARC_TEST_PATH"]

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=TTT_EPOCHS,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=TTT_LR,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    def load_model(path):
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=path,
            full_finetuning=False,
            load_in_4bit=False,
            local_files_only=True,
            use_gradient_checkpointing=False,
            max_seq_length=max_seq_length,
        )

        model = FastLanguageModel.get_peft_model(model, **peft_params)

        for name, param in model.named_parameters():
            if param.dtype == torch.float32:
                param.data = param.data.to(torch.bfloat16)

        default_weights = get_peft_model_state_dict(model, adapter_name="default")
        default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

        collator = QwenDataCollatorForCompletionOnlyLM(
            tokenizer=tokenizer,
            mlm=False,
        )

        formatter = QwenFormatter(tokenizer=tokenizer)

        return model, tokenizer, default_weights, collator, formatter, formatter.max_new_tokens()

    model, tokenizer, default_weights, collator, formatter, max_new_tokens = load_model(model_path)
    second_model_path = os.environ.get("ARC_SECOND_MODEL_PATH")  # v16: the second pass's generator

    max_score = -np.log(0.2)

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "../inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    sec_per_token = None  # measured training speed, set after this worker's first training
    top_env = os.environ.get("ARC_TOP_COPIES", "16")  # 24 trains more where time allows
    share_16 = AUTO_SHARE_16 if top_env == "auto" else None
    top_copies = 16 if top_env == "auto" else int(top_env)

    # v15 second pass: a task is eligible once its first pass is done; a claim file keeps two
    # GPUs from taking the same task
    dir_done, dir_claims = "../pass1_done", "../pass2_claims"
    os.makedirs(dir_done, exist_ok=True)
    os.makedirs(dir_claims, exist_ok=True)

    def top_votes(key):
        """Fewest votes behind any of this task's test outputs' top answer (0: no guess)."""
        n_tests = len(arc_test_set.queries[key]["test"])
        votes = [defaultdict(int) for _ in range(n_tests)]
        for name in os.listdir(dir_outputs):
            bk = name.split(".")[0]
            if bk.rsplit("_", 1)[0] != key:
                continue
            with bz2.BZ2File(os.path.join(dir_outputs, name)) as f:
                for g in pickle.load(f):
                    votes[int(bk.rsplit("_", 1)[1])][tuple(map(tuple, g["solution"]))] += 1
        return min(max(v.values(), default=0) for v in votes)

    def task_cells(key):
        q = arc_test_set.queries[key]
        return sum(len(g) * len(g[0]) for p in q["train"] for g in p.values()) + \
            sum(len(t["input"]) * len(t["input"][0]) for t in q["test"])

    def claim_weak_task():
        """Smallest finished task with a weak answer that no GPU has claimed yet, or None."""
        done = [k for k in os.listdir(dir_done) if not os.path.exists(os.path.join(dir_claims, k))]
        for key in sorted(done, key=task_cells):
            votes = top_votes(key)
            if votes >= SECOND_PASS_MAX_VOTES:
                continue
            try:
                os.close(os.open(os.path.join(dir_claims, key), os.O_CREAT | os.O_EXCL | os.O_WRONLY))
            except FileExistsError:
                continue
            print(f"[Rank {rank}] second pass for {key} (top answer has {votes} votes)")
            return key
        return None

    def jobs():
        """(task, training seed, view seed, second pass): the queue, then v15 second passes."""
        while not queue.empty():
            if time.time() > end_time:
                print(f"[Rank {rank}] stop!")
                return
            key = queue.get()
            if key is None:
                break
            yield key, 1, 2, False
        while SECOND_PASS and time.time() < end_time - MIN_DECODE_SECONDS:
            key = claim_weak_task()
            if key is None:
                if len(os.listdir(dir_done)) >= len(arc_test_set.keys):
                    return
                time.sleep(30)  # other GPUs are still on their last first-pass tasks
                continue
            yield key, 7, 3, True

    for key, train_seed, view_seed, second in jobs():

        if second and second_model_path:
            # v16: swap in a separately trained base model once, so the second pass's errors differ
            # from the first's (a second adapter on the same base model found nothing new, loop 7)
            print(f"[Rank {rank}] loading {second_model_path} for the second pass")
            try:
                del model
                gc.collect()
                torch.cuda.empty_cache()
                model, tokenizer, default_weights, collator, formatter, max_new_tokens = load_model(second_model_path)
            except Exception as e:  # never retry: a crash restart would reload and fail again
                print(f"[Rank {rank}] second model failed to load, stopping this GPU: {e!r}")
                return
            second_model_path = None
            sec_per_token = None  # training speed differs between the two models

        start_time = time.time()
        # a second pass has the rest of the clock: the queue is empty
        budget = end_time - start_time if second else task_budget(queue, n_workers, end_time)
        task_deadline = min(end_time, start_time + budget)
        print(f"[Rank {rank}] puzzle {key}: budget {budget:.0f}s")

        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        def train_list(n):
            ds = puzzle_ds.augment(n=n, shfl_keys=True, seed=train_seed)
            return ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length).as_list(formatter)

        # v7: size test-time training to the time this task can afford
        samples = train_list(16)
        tokens_16 = sum(len(tokenizer.encode(s["text"])) for s in samples)
        n_copies = plan_training(budget, end_time - time.time(), tokens_16, sec_per_token, top_copies, share_16)
        print(f"[Rank {rank}] puzzle {key}: training on {n_copies} copies ({tokens_16} tokens at 16)")
        if second and not n_copies:
            print(f"[Rank {rank}] second pass for {key} skipped: training does not fit")
            continue
        if n_copies not in (0, 16):
            samples = train_list(n_copies)

        stats = "skipped (decoding with the base model)"
        if n_copies:
            train_start = time.time()
            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):

                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(samples),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )

                stats = trainer.train()

                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

                del trainer
            # keep the slower of the last two measurements so estimates err long
            rate = (time.time() - train_start) / max(1, tokens_16 * n_copies / 16)
            sec_per_token = rate if sec_per_token is None else max(rate, 0.5 * (rate + sec_per_token))

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        # v4: a task whose training overran its budget still gets decoded (it used to get nothing)
        task_deadline = min(end_time, max(task_deadline, time.time() + MIN_DECODE_SECONDS))

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        def view_batches(seed):
            eval_ds = puzzle_ds_multi.augment(n=2, seed=seed)
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

            test_id_to_subkeys = defaultdict(list)
            for subkey in sorted(eval_ds.keys):
                test_id = subkey.split(".")[0].split("_")[1]
                test_id_to_subkeys[test_id].append(subkey)

            batches = []
            for test_id, subkeys in test_id_to_subkeys.items():
                # 0: permute x 2
                # 4: rot90.rot90.permute x 2
                batch = []
                for offset in [0, 4]:
                    batch.extend(subkeys[offset:offset+2])
                batches.append(batch)
                # 2: permute.rot90 x 2
                # 6: rot90.rot90.rot90.permute x 2
                batch = []
                for offset in [2, 6]:
                    batch.extend(subkeys[offset:offset+2])
                batches.append(batch)
            for test_id, subkeys in test_id_to_subkeys.items():
                # 8: transpose.permute x 2
                # 12: transpose.rot90.rot90.permute x 2
                batch = []
                for offset in [8, 12]:
                    batch.extend(subkeys[offset:offset+2])
                batches.append(batch)
                # 10: transpose.rot90.permute x 2
                # 14: transpose.rot90.rot90.rot90.permute x 2
                batch = []
                for offset in [10, 14]:
                    batch.extend(subkeys[offset:offset+2])
                batches.append(batch)
            return eval_ds, test_id_to_subkeys, batches

        eval_ds, test_id_to_subkeys, batches = view_batches(seed=view_seed)

        with torch.inference_mode():

            known_scores = {}
            covered = set()  # test outputs ("<task>_<i>") with at least one saved guess

            def decode_batch(subkeys, score_limit, deadline):
                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, score_limit, deadline)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                            "loose": score_limit != max_score,  # from the looser fallback pass
                            "second": second,  # from the v15 second adapter
                        })

                    if len(decoded_result):
                        covered.add(bk)
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

            for subkeys in batches:

                spend_time = time.time() - start_time
                if time.time() > task_deadline:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")
                decode_batch(subkeys, max_score, task_deadline)

            # v10: a test output with no guess above the usual 20% cutoff gets one looser pass over
            # 4 of its views, instead of falling back to a placeholder grid. The views must share the
            # grid shape (identity and rot180, as in the first batch above): a batch mixing rot90 views
            # has unequal prompt lengths and crashed the worker in loop 2
            for test_id, subkeys in test_id_to_subkeys.items():
                bk = f"{key}_{test_id}"
                fallback_deadline = min(end_time, time.time() + FALLBACK_DECODE_SECONDS)
                if second or bk in covered or time.time() >= fallback_deadline - 10:
                    continue
                print(f"[Rank {rank}] no guess for {bk}; looser decode")
                try:
                    decode_batch(subkeys[0:2] + subkeys[4:6], FALLBACK_MAX_SCORE, fallback_deadline)
                except Exception as e:  # a lost fallback must never cost the rest of the queue
                    print(f"[Rank {rank}] looser decode failed for {bk}: {e!r}")
                    torch.cuda.empty_cache()

            # v13: time left in this task's budget goes to 16 more views (new color permutations and
            # example orders), which adds candidates and votes; loop 4 found the answer among the
            # candidates for 13/24 tasks and runs ended ~20% early. Last, because eval_ds changes
            # and decode_batch reads it
            if EXTRA_VIEWS and not second and time.time() < task_deadline - EXTRA_VIEWS_MIN_SECONDS:
                try:
                    eval_ds, _, extra_batches = view_batches(seed=3)
                    for subkeys in extra_batches:
                        if time.time() > task_deadline:
                            break
                        print(f"[Rank {rank}] extra views {subkeys}")
                        decode_batch(subkeys, max_score, task_deadline)
                except Exception as e:  # extra views are optional; never lose the task over them
                    print(f"[Rank {rank}] extra views failed for {key}: {e!r}")
                    torch.cuda.empty_cache()

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")
        if not second:
            open(os.path.join(dir_done, key), "w").close()

In [ ]:
%%writefile starter.py
"""Spawn one test-time-training worker per GPU over a shared task queue.

Adapted from NVARC's ARC Prize 2025 starter: paths come from the environment,
tasks are queued smallest-first so the dynamic per-task budget in
llm_worker.task_budget can hand any time saved on easy tasks to later ones,
and ARC_TASK_KEYS can restrict the run to a subset (smoke tests).
"""
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time, n_workers):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"../worker{rank-1}"):
            time.sleep(5)

    from llm_worker import worker

    with open(f"../worker{rank}", "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")

    # an exception in one worker makes mp.spawn stop all of them, so the rest of the queue would go
    # unsolved: log it, free the GPU and restart (only the task in progress is lost)
    import gc
    import traceback
    for attempt in range(4):
        try:
            worker(rank, queue, end_time, n_workers)
            break
        except Exception:
            traceback.print_exc()
        # outside the except block, so the traceback no longer holds the crashed model
        gc.collect()
        torch.cuda.empty_cache()
        if time.time() > end_time - 300:
            break
        print(f"[Rank {rank}] restarting after a crash")

    print(f"[Rank {rank}] done!")


def task_size(task):
    return sum(len(g) * len(g[0]) for pair in task["train"] for g in pair.values()) + \
        sum(len(t["input"]) * len(t["input"][0]) for t in task["test"])


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    with open(os.environ["ARC_TEST_PATH"], "r") as f:
        data = json.load(f)

    keys = sorted(data.keys(), key=lambda k: (task_size(data[k]), k))
    if os.getenv("ARC_TASK_ORDER") == "file":  # v17 NVARC-faithful mode: the input file's order
        keys = list(data.keys())
    subset = os.getenv("ARC_TASK_KEYS")
    if subset:
        keys = [k for k in keys if k in set(subset.split(","))]

    n_workers = max(1, torch.cuda.device_count())
    print(f"Queueing {len(keys)} tasks on {n_workers} GPUs")

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(n_workers):
        queue.put(None)

    mp.spawn(local_worker, args=(queue, args.end_time, n_workers), nprocs=n_workers)


In [ ]:
%%writefile merge.py
"""Build submission.json from the LLM's ranked guesses plus the rule solver.

  python merge.py symbolic   # run the rule solver, write a safety submission.json
  python merge.py final      # merge LLM outputs (../inference_outputs) on top

Ranking: the LLM's top two guesses come first (NVARC's kgmon selection).
Rule-solver programs that reproduce every training pair only fill slots the
LLM left empty: on the ARC-AGI-2 public eval they fit some tasks without
generalising, so they must never displace an LLM guess.
"""
import json
import os
import sys
from multiprocessing import Pool

import numpy as np

TEST_PATH = os.environ.get("ARC_TEST_PATH", "")
SOLUTIONS_PATH = os.environ.get("ARC_SOLUTIONS_PATH", "")
SYMBOLIC_PATH = os.environ.get("ARC_SYMBOLIC_PATH", "symbolic_candidates.json")
LLM_DIR = os.environ.get("ARC_LLM_DIR", "../inference_outputs")
OUT_PATH = os.environ.get("ARC_SUBMISSION_PATH", "submission.json")
RULE_SECOND = os.environ.get("ARC_RULE_SECOND", "1") == "1"


def valid_grid(g):
    try:
        a = np.asarray(g)
    except Exception:
        return False
    return a.ndim == 2 and 0 < a.shape[0] <= 30 and 0 < a.shape[1] <= 30 and \
        np.issubdtype(a.dtype, np.integer) and a.min() >= 0 and a.max() <= 9


def build_submission(challenges, llm_ranked, symbolic):
    """challenges: {task_id: task}; llm_ranked: {"<task_id>_<i>": [grid, ...]} best first;
    symbolic: {task_id: [[grid, ...] per test input]}."""
    sub = {}
    for tid, task in challenges.items():
        entries = []
        for i, t in enumerate(task["test"]):
            cands = []

            def add(g):
                if valid_grid(g):
                    g = np.asarray(g).tolist()
                    if g not in cands:
                        cands.append(g)

            llm = llm_ranked.get(f"{tid}_{i}", [])[:2]
            sym = symbolic.get(tid, [])
            sym = sym[i] if i < len(sym) else []
            # a rule program that reproduces every train pair (97% precise on public training
            # tasks) outranks the model's second guess, but never its first; v17 NVARC-faithful
            # mode (ARC_RULE_SECOND=0) keeps both model guesses and only fills empty slots
            if RULE_SECOND:
                order = llm[:1] + sym[:1] + llm[1:] + sym[1:]
            else:
                order = llm + sym
            for g in order:
                add(g)
            add(t["input"])
            add([[0]])
            entries.append({"attempt_1": cands[0], "attempt_2": cands[1]})
        sub[tid] = entries
    return sub


def score(sub, solutions):
    total = 0.0
    for tid, outs in solutions.items():
        for i, truth in enumerate(outs):
            e = sub[tid][i]
            if e["attempt_1"] == truth or e["attempt_2"] == truth:
                total += 1 / len(outs)
    return total, len(solutions)


def _symbolic_one(item):
    import arc_solver
    tid, task = item
    try:
        return tid, arc_solver.verified_candidates(task, time_budget=20.0)[1]
    except Exception:
        return tid, [[] for _ in task["test"]]


def run_symbolic(challenges, processes=4):
    with Pool(processes) as pool:
        return dict(pool.map(_symbolic_one, sorted(challenges.items()), chunksize=2))


def load_llm_ranked(test_path, llm_dir):
    if not os.path.isdir(llm_dir) or not os.listdir(llm_dir):
        return {}
    from arc_decoder import ArcDecoder, score_kgmon
    decoder = ArcDecoder(dataset=None, n_guesses=2)
    decoder.load_decoded_results(llm_dir)
    return decoder.run_selection_algo(score_kgmon)


def diagnose(challenges, llm_dir, solutions):
    """Commit runs only: is the answer lost in generation (never among the candidates) or in
    selection (a candidate, but not in the top two)? Also scores each selection rule."""
    from arc_decoder import ArcDecoder, score_kgmon, selection_algorithms
    decoder = ArcDecoder(dataset=None, n_guesses=2)
    decoder.load_decoded_results(llm_dir)
    found, loose_only, hits = 0.0, 0, []
    for tid, outs in solutions.items():
        for i, truth in enumerate(outs):
            cands = decoder.decoded_results.get(f"{tid}_{i}", {}).values()
            right = [c for c in cands if np.array_equal(c["solution"], truth)]
            if right:
                found += 1 / len(outs)
                hits.append(f"{tid}_{i}:{len(right)}/{len(cands)}")
                loose_only += all(c.get("loose") for c in right)
    print(f"*** answer among the candidates: {found:.2f}/{len(solutions)} "
          f"({loose_only} only from the looser pass) {' '.join(hits)}")
    for algo in selection_algorithms:
        s, n = score(build_submission(challenges, decoder.run_selection_algo(algo), {}), solutions)
        print(f"*** selection {algo.__name__}: {s:.2f}/{n}")
    # v15: the same run without the second adapter's candidates, so its effect is measured on
    # identical first-pass results (no run-to-run noise)
    second = sum(c.get("second", False) for v in decoder.decoded_results.values() for c in v.values())
    if second:
        tasks = sorted({bk.rsplit("_", 1)[0] for bk, v in decoder.decoded_results.items()
                        if any(c.get("second") for c in v.values())})
        decoder.decoded_results = {bk: {k: c for k, c in v.items() if not c.get("second")}
                                   for bk, v in decoder.decoded_results.items()}
        s, n = score(build_submission(challenges, decoder.run_selection_algo(score_kgmon), {}), solutions)
        print(f"*** first pass only (kgmon): {s:.2f}/{n}; second pass added {second} candidates "
              f"on {len(tasks)} tasks: {' '.join(tasks)}")


def write(sub, challenges):
    assert set(sub) == set(challenges), "every task id must be present"
    with open(OUT_PATH, "w") as f:
        json.dump(sub, f)
    if SOLUTIONS_PATH and os.path.exists(SOLUTIONS_PATH):
        sol = json.load(open(SOLUTIONS_PATH))
        sol = {k: v for k, v in sol.items() if k in challenges}
        s, n = score(sub, sol)
        print(f"*** local score: {s:.2f}/{n} = {100 * s / max(n, 1):.2f}%")


if __name__ == "__main__":
    challenges = json.load(open(TEST_PATH))
    mode = sys.argv[1]
    if mode == "symbolic":
        symbolic = run_symbolic(challenges)
        json.dump(symbolic, open(SYMBOLIC_PATH, "w"))
        n = sum(any(c for c in v) for v in symbolic.values())
        print(f"rule solver: programs found for {n}/{len(challenges)} tasks")
        write(build_submission(challenges, {}, symbolic), challenges)
    elif mode == "final":
        symbolic = json.load(open(SYMBOLIC_PATH)) if os.path.exists(SYMBOLIC_PATH) else {}
        llm = load_llm_ranked(TEST_PATH, LLM_DIR)
        print(f"LLM produced guesses for {len(llm)} test outputs")
        write(build_submission(challenges, llm, symbolic), challenges)
        if llm and SOLUTIONS_PATH and os.path.exists(SOLUTIONS_PATH):
            sol = json.load(open(SOLUTIONS_PATH))
            try:  # a diagnostic must never fail the run
                diagnose(challenges, LLM_DIR, {k: v for k, v in sol.items() if k in challenges})
            except Exception as e:
                print("diagnostic failed:", repr(e))
    else:
        raise SystemExit(f"unknown mode {mode}")


In [ ]:
# smoke-test subset for commit runs: the N smallest public eval tasks, or N random ones
if not RERUN:
    import random
    from starter import task_size
    data = json.load(open(os.environ["ARC_TEST_PATH"]))
    n = int(os.environ["ARC_SMOKE_TASKS"])
    if os.environ["ARC_SMOKE_MODE"] == "realistic":
        keys = random.Random(SMOKE_SEED).sample(sorted(data), n)
    else:
        keys = sorted(data, key=lambda k: task_size(data[k]))[:n]
    sub_path = "smoke_challenges.json"
    json.dump({k: data[k] for k in keys}, open(sub_path, "w"))
    os.environ["ARC_TEST_PATH"] = os.path.abspath(sub_path)
    print("smoke tasks:", keys)

In [ ]:
# 1) rule solver on CPU, in the background at low priority so the GPUs start at once
# (it took up to ~20 min of idle GPU time when it ran first)
import sys
sym_proc = subprocess.Popen(["nice", "-n", "19", sys.executable, "merge.py", "symbolic"],
                            stdout=open("symbolic.log", "w"), stderr=subprocess.STDOUT)

In [ ]:
# 2) NVARC test-time training + DFS decoding on all GPUs
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
# 3) merge LLM guesses with the rule solver's programs; the rule solver must be done first,
# or its own submission.json would overwrite this one
try:
    sym_proc.wait(timeout=300)
except subprocess.TimeoutExpired:
    sym_proc.kill(); sym_proc.wait()
    print("rule solver stopped unfinished; merging without it")
print(open("symbolic.log").read()[-500:])
!python merge.py final

In [ ]:
!ls -la /kaggle/working/ && rm -f ../worker*